# Corpus and retrieval walkthrough

What this notebook is for: showing what the pipeline actually does to a corpus,
and where its retrieval quality comes from. It runs entirely offline on the
bundled evaluation corpus.

The measured results this repository publishes are in `docs/results.md`, produced
by `scripts/run_experiment.py`. Nothing here supersedes them; this is the
exploratory view that sits behind them.

In [1]:
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag_platform.config import PipelineConfig
from rag_platform.data.local_documents import load_directory
from rag_platform.logging_utils import configure_logging
from rag_platform.retrieval.index import RetrievalIndex
from rag_platform.utils.seeds import set_global_seed

configure_logging("WARNING")

ROOT = Path.cwd().parent
config = PipelineConfig.from_yaml(ROOT / "configs" / "default.yaml")
set_global_seed(config.run.seed)

documents = load_directory(ROOT / config.corpus.source_dir)
print(f"{len(documents)} documents")
print(Counter(d.source_type.value for d in documents))

20 documents
Counter({'biomedical_abstract': 10, 'sec_filing': 8, 'user_document': 2})


## 1. What ingestion preserves

The point of section-aware ingestion is that a filing is not prose. Segmenting on
Item headings means a citation can name *Item 1A. Risk Factors* rather than a
chunk id, and means a risk factor is not split across two chunks.

In [2]:
filing = next(d for d in documents if d.metadata.source_file == "northwind_energy_10k_fy2024.md")
print(filing.metadata.company_name, filing.metadata.form_type, filing.metadata.fiscal_year)
print(f"CIK {filing.metadata.cik}, {len(filing.text.split())} words\n")

for section in filing.sections:
    print(f"  {len(section.text.split()):5d} words  {section.heading}")

Northwind Energy Corporation 10-K 2024
CIK 0001000101, 799 words

    153 words  Item 1. Business
    279 words  Item 1A. Risk Factors
    254 words  Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
     82 words  Item 8. Financial Statements and Supplementary Data


In [3]:
abstract = next(d for d in documents if d.metadata.source_file == "gene_editing_sickle_cell.md")
print(abstract.metadata.journal, abstract.metadata.publication_year, "| PMID", abstract.metadata.pmid)
print("MeSH:", abstract.metadata.mesh_terms)
print("sections:", [s.heading for s in abstract.sections])

J Haematol Ther 2024 | PMID 40010001
MeSH: ['Anemia, Sickle Cell', 'Gene Editing', 'Hematopoietic Stem Cell Transplantation']
sections: ['Background', 'Methods', 'Results', 'Conclusions']


## 2. Chunking

Chunk length is the most consequential retrieval knob in this pipeline. The
sweep in `docs/results.md` section 6 shows why comparing strategies at a fixed
*k* gives the wrong answer: larger chunks return more text, so they are credited
with more of the judged material regardless of ranking quality.

In [4]:
from rag_platform.features.chunking import chunk_documents
from rag_platform.features.metadata import enrich_metadata

enriched = [enrich_metadata(d) for d in documents]

for strategy in ("section_aware", "fixed_window"):
    variant = config.with_overrides({"chunking.strategy": strategy})
    chunks = chunk_documents(enriched, variant.chunking)
    lengths = [c.token_count for c in chunks]
    print(
        f"{strategy:15s} {len(chunks):3d} chunks  "
        f"mean {sum(lengths)/len(lengths):5.1f}  "
        f"min {min(lengths):3d}  max {max(lengths):3d}  "
        f"with heading {sum(1 for c in chunks if c.section_heading):3d}"
    )

section_aware    77 chunks  mean  76.5  min  20  max 279  with heading  77
fixed_window     29 chunks  mean 226.8  min  42  max 300  with heading   0


## 3. Where lexical and dense retrieval disagree

This is the case for hybrid retrieval, made concrete. Lexical matching wins when
the query names an exact identifier; dense matching wins on paraphrase. Neither
wins everywhere, which is why both run.

In [5]:
index = RetrievalIndex(config)
index.build_from_documents(documents)
index.load_reranker(ROOT / config.run.output_dir / "reranker.joblib")
print(index.describe())

{'built': True, 'documents': 20, 'chunks': 77, 'sources': {'biomedical_abstract': 40, 'user_document': 9, 'sec_filing': 28}, 'embedding_backend': 'tfidf_svd', 'embedding_dim': 76, 'fusion': 'weighted', 'reranking': True, 'mean_chunk_tokens': 76.50649350649351, 'build_ms': 332.72}


In [6]:
def compare(query, k=3):
    print(f"\nQ: {query}")
    lex = index._hybrid.lexical.search(query, k)
    den = index._hybrid.dense.search(query, k)
    chunks = index.chunks
    print("  lexical (BM25)")
    for i, s in lex:
        print(f"    {s:6.2f}  {chunks[i].metadata.source_file:42s} {chunks[i].section_heading}")
    print("  dense (latent semantic indexing)")
    for i, s in den:
        print(f"    {s:6.2f}  {chunks[i].metadata.source_file:42s} {chunks[i].section_heading}")

# An exact identifier: lexical matching should dominate.
compare("Meridian Semiconductor 65 nanometre node capacity reallocation")

# A paraphrase with little vocabulary overlap: dense matching has more to offer.
compare("did the published biomarker signature hold up when tested somewhere else")


Q: Meridian Semiconductor 65 nanometre node capacity reallocation
  lexical (BM25)
     18.99  meridian_semiconductor_10q_q2_fy2025.md    Item 1A. Risk Factors
      6.91  meridian_semiconductor_10k_fy2024.md       Item 1. Business
      2.98  meridian_semiconductor_10k_fy2024.md       Item 1A. Risk Factors
  dense (latent semantic indexing)
      0.93  meridian_semiconductor_10q_q2_fy2025.md    Item 1A. Risk Factors
      0.32  meridian_semiconductor_10k_fy2024.md       Item 1. Business
      0.06  cascade_logistics_10k_fy2023.md            Item 1A. Risk Factors

Q: did the published biomarker signature hold up when tested somewhere else
  lexical (BM25)
      8.38  gut_microbiome_response.md                 Conclusions
      6.90  gut_microbiome_response.md                 Methods
      4.65  gut_microbiome_response.md                 Results
  dense (latent semantic indexing)
      0.53  gut_microbiome_response.md                 Methods
      0.50  gut_microbiome_response.md      

## 4. Fusion and reranking

Weighted fusion normalises each retriever's scores over the candidate pool and
combines them; the reranker then re-scores the short list with features fusion
cannot see. On the held-out split the reranker moves NDCG@5 from 0.651 to 0.760
while barely moving recall — it reorders a good candidate set rather than
finding chunks fusion missed.

In [7]:
query = "Which turbine supplier concentration risk does Northwind Energy disclose?"

print("fused only")
for r in index.search(query, top_k=5, rerank=False):
    print(f"  {r.score:5.3f} lex={r.lexical_score:6.2f} den={r.dense_score:5.2f}  "
          f"{r.chunk.metadata.source_file:36s} {r.chunk.section_heading}")

print("\nafter reranking")
for r in index.search(query, top_k=5, rerank=True):
    score = r.rerank_score if r.rerank_score is not None else r.score
    print(f"  {score:5.3f}  {r.chunk.metadata.source_file:36s} {r.chunk.section_heading}")

fused only
  0.860 lex=  6.98 den= 0.71  northwind_energy_10k_fy2023.md       Item 1. Business
  0.829 lex= 10.13 den= 0.49  northwind_energy_10k_fy2024.md       Item 1A. Risk Factors
  0.734 lex=  6.09 den= 0.60  northwind_energy_10k_fy2024.md       Item 1. Business
  0.584 lex=  8.91 den= 0.24  northwind_energy_10k_fy2023.md       Item 1A. Risk Factors
  0.405 lex=  4.22 den= 0.28  atlas_payments_10k_fy2024.md         Item 1A. Risk Factors

after reranking
  0.980  northwind_energy_10k_fy2024.md       Item 1A. Risk Factors
  0.875  northwind_energy_10k_fy2023.md       Item 1. Business
  0.874  northwind_energy_10k_fy2024.md       Item 1. Business
  0.688  northwind_energy_10k_fy2023.md       Item 1A. Risk Factors
  0.617  atlas_payments_10k_fy2024.md         Item 1A. Risk Factors


## 5. Entity scoping

Two sub-queries of a comparison share almost every content word, so term and
vector scores alone do not keep each one's evidence in the right document. An
early version of this pipeline answered a question about Northwind Energy by
quoting a different company's revenue — with a groundedness score of 1.000,
because the sentences were faithfully quoted from the wrong document.

In [8]:
for entity in (None, "Northwind Energy"):
    results = index.search("revenue growth in 2024", top_k=5, entity=entity)
    companies = {r.chunk.metadata.company_name for r in results}
    print(f"entity={entity!r:20s} -> {sorted(c for c in companies if c)}")

entity=None                 -> ['Atlas Payments Holdings plc', 'Cascade Logistics Group, Inc.', 'Northwind Energy Corporation', 'Vertexa Biopharma Corporation']
entity='Northwind Energy'   -> ['Northwind Energy Corporation']


## 6. A full answer

The orchestrator plans, retrieves, synthesises and verifies. Every citation
carries the quote it was derived from and a flag set by checking that quote
against the chunk it points at.

In [9]:
from rag_platform.agents.orchestrator import Orchestrator

orchestrator = Orchestrator(index, config)
answer = orchestrator.answer("Compare Northwind Energy revenue growth in 2024 against 2023", top_k=6)

print("sub-queries:", answer.subqueries)
print("strategy:   ", answer.diagnostics["strategy"], "| tools:", answer.tools_used)
print()
print(answer.text)
print()
for c in answer.citations:
    print(f"  [{c.marker}] verified={c.verified} support={c.support_score:.2f}  {c.label}")
print(f"\ngroundedness={answer.groundedness:.2f}  latency={answer.latency_ms:.1f}ms")

2026-09-15T04:36:59.416981Z [info     ] query_decomposed               parts=2 strategy=temporal_comparison


2026-09-15T04:36:59.460856Z [info     ] query_answered                 evidence=4 groundedness=1.0 latency_ms=43.95 strategy=temporal_comparison subqueries=2


sub-queries: ['Northwind Energy revenue growth in 2024', 'Northwind Energy revenue growth in 2023']
strategy:    temporal_comparison | tools: ['financial_lookup']

Northwind Energy Corporation develops, owns and operates onshore wind and utility-scale solar generation assets. As of December 31, 2023 the Company owned interests in 31 operating projects with 3,760 megawatts of net installed capacity, of which 2,940 megawatts is wind and 820 megawatts is solar. [1] Approximately 81 percent of expected 2024 generation was contracted under power purchase agreements. The Company employed 1,155 people at December 31, 2023. [1] The Company's development pipeline totalled 6,900 megawatts at year end, of which 1,450 megawatts had reached the advanced stage, meaning interconnection queue position, site control and a signed offtake term sheet were all in place. Northwind employed 1,240 people at December 31, 2024, of whom 410 worked in field operations and maintenance. [2] Total revenue for 2024 w

## 7. Retrieval quality by query type

The aggregate hides the system's clearest weakness. Comparison queries are far
harder than single-hop lookups, and averaging the two together conceals it.

In [10]:
from rag_platform.evaluation.harness import evaluate_retrieval
from rag_platform.evaluation.qrels import load_qrels

qrels = load_qrels(ROOT / config.corpus.qrels_path)
result = evaluate_retrieval(index, qrels)

print(f"all {result['queries']} judged queries")
for name in ("recall@budget", "recall@5", "precision@5", "ndcg@5", "reciprocal_rank"):
    print(f"  {name:<18} {result['metrics'][name]:.3f}")

print("\nby query type")
for qtype, metrics in result["by_query_type"].items():
    print(f"  {qtype:12s} n={metrics['queries']:.0f}  "
          f"recall@budget={metrics['recall@budget']:.3f}  ndcg@5={metrics['ndcg@5']:.3f}")

all 38 judged queries
  recall@budget      0.803
  recall@5           0.912
  precision@5        0.263
  ndcg@5             0.808
  reciprocal_rank    0.822

by query type
  comparison   n=6  recall@budget=0.333  ndcg@5=0.655
  single_hop   n=32  recall@budget=0.891  ndcg@5=0.837


These figures cover all 38 judged queries, including the 19 the reranker was
fitted on, so they are optimistic. **The numbers to compare configurations by
are the held-out ones in `docs/results.md`**, together with their bootstrap
intervals — which, at 19 queries, are wide enough that most of the differences
between configurations are not separated.